# 15 — MLP WJ-Native (train + index in Weighted Jaccard space)

**Advisor experiment:** Same global MLP compressor, but embeddings are **non-negative + L1-normalized** (probability simplex), trained with **WJ triplet loss**, indexed with **HNSW + WeightedJaccard** (train/index metric aligned).

**Pipeline:**
```
quadtree → log1p (full only) → MLP → ReLU → L1 norm → 512-D
         → HNSW WeightedJaccard → optional exact raw-WJ rerank
```

**Prereqs:** Run `00_cache_data.ipynb` first. On Orion, 10k encodings live at:
`/mnt/data1/shapeSimilarity/encodings/pk-real10k0.002` → cached as `/tmp/qt_10k.npy`.
GT: `/mnt/data1/shapeSimilarity/warehouse/pk-query-10k` → `/tmp/gt_lookup_10k.pkl`.

**Cosine warm-start (Orion):** `/mnt/data1/ruban/hpmlproj/best_model/best_compressor_v1_clean.pt` (10k), `best_compressor_full_fixed.pt` (full)  
**WJ-native outputs:** `/tmp/best_compressor_wj_native_10k.pt`, `/tmp/best_compressor_wj_native_full.pt`  
**Results:** `/tmp/results_mlp_wj_native.pkl`

In [9]:
# ── Configuration (edit here) ──────────────────────────────────────────────
dataset_name = "10k"          # "10k" or "full"
run_training = True           # False = load checkpoint only, run eval
run_eval = True
device_str = "cuda:0"         # set "cpu" if no GPU

THREADS = 32
seed = 42

QUERY_START_10K = 8000
QUERY_START_FULL = 187019

# Training (10k defaults from presentation; full uses fewer epochs if init from fixed ckpt)
max_pos = 30
batch_size = 512
epochs_10k = 50
epochs_full = 50
lr = 1e-3
weight_decay = 1e-4
wj_margin = 0.3
init_from_cosine_ckpt = True   # warm-start encoder from existing MLP checkpoints

# Retrieval
candidate_ks_10k = [500, 1000]
candidate_ks_full = [1000, 2000]
rerank_mode = "gpu"              # "gpu" or "dense"
rerank_batch_size = 16

out_path = "/tmp/results_mlp_wj_native.pkl"

# Pretrained cosine MLP checkpoints (DGX → copied to Orion)
BEST_MODEL_DIR = "/mnt/data1/ruban/hpmlproj/best_model"

In [10]:
import gc
import os
import pickle
import random
import time
from pathlib import Path

import nmslib
import numpy as np
import psutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

device = torch.device(device_str if torch.cuda.is_available() else "cpu")

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

print(f"device={device} | dataset={dataset_name}")

device=cuda:0 | dataset=10k


In [11]:
# ── Model: MLP → WJ-valid simplex (Buthi / professor pipeline) ─────────────
class QuadtreeCompressorWJ(nn.Module):
    """Global MLP; output is ReLU + L1 normalize for Weighted Jaccard index/train."""

    def __init__(self, in_dim, out_dim=512, use_log1p=False):
        super().__init__()
        self.use_log1p = use_log1p
        self.net = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False), nn.BatchNorm1d(out_dim),
        )

    def _preprocess(self, x):
        if self.use_log1p:
            x = torch.log1p(x * 1e6)
        return x

    def forward_raw(self, x):
        """Linear trunk only (for loading cosine checkpoints)."""
        return self.net(self._preprocess(x))

    def forward(self, x):
        out = self.forward_raw(x)
        out = F.relu(out)
        out = out / out.sum(dim=1, keepdim=True).clamp(min=1e-10)
        return out


def wj_similarity(a, b):
    """Weighted Jaccard: sum(min) / sum(max) on last dim."""
    mins = torch.minimum(a, b).sum(dim=-1)
    maxs = torch.maximum(a, b).sum(dim=-1).clamp(min=1e-10)
    return mins / maxs


def wj_triplet_loss(anchors, positives, margin=0.3):
    """
    WJ triplet on embedding simplex (same as 03_neural_minhash).
    Loss = relu(sim_hard_neg - sim_pos + margin); only violated pairs.
    """
    sim_ap = wj_similarity(anchors, positives)

    mins_cross = torch.min(anchors.unsqueeze(1), positives.unsqueeze(0)).sum(dim=2)
    maxs_cross = torch.max(anchors.unsqueeze(1), positives.unsqueeze(0)).sum(dim=2)
    sim_cross = mins_cross / maxs_cross.clamp(min=1e-10)
    sim_cross.fill_diagonal_(-1e9)
    sim_an = sim_cross.max(dim=1).values

    loss = F.relu(sim_an - sim_ap + margin)
    violated = loss > 0
    if violated.sum() == 0:
        z = torch.tensor(0.0, device=anchors.device, requires_grad=True)
        return z, 0, sim_ap.detach().mean(), sim_an.detach().mean()
    return loss[violated].mean(), int(violated.sum().item()), sim_ap.detach().mean(), sim_an.detach().mean()


class AnchorPositiveDataset(Dataset):
    def __init__(self, qtree_vectors, gt_lookup, query_start, max_pos=30):
        self.vecs = torch.tensor(qtree_vectors, dtype=torch.float32)
        self.pairs = []
        for qid, neighbors in gt_lookup.items():
            for nid in neighbors[:max_pos]:
                if nid < query_start:
                    self.pairs.append((qid, nid))
        random.shuffle(self.pairs)
        print(f"Anchor-positive pairs: {len(self.pairs):,}")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        qid, pos_id = self.pairs[idx]
        return self.vecs[qid], self.vecs[pos_id]


def weighted_jaccard_np(a, b):
    mins = np.minimum(a, b).sum()
    maxs = np.maximum(a, b).sum()
    return float(mins / max(maxs, 1e-10))


print("Model + WJ triplet loss defined.")

Model + WJ triplet loss defined.


In [12]:
# ── Data loading ─────────────────────────────────────────────────────────────
def load_dataset(name):
    if name == "10k":
        qt = np.load("/tmp/qt_10k.npy")
        with open("/tmp/gt_lookup_10k.pkl", "rb") as f:
            gt = pickle.load(f)
        query_start = QUERY_START_10K
        use_log1p = False
        cosine_ckpt = f"{BEST_MODEL_DIR}/best_compressor_v1_clean.pt"
        wj_ckpt = "/tmp/best_compressor_wj_native_10k.pt"
        candidate_ks = candidate_ks_10k
        n_epochs = epochs_10k
    elif name == "full":
        qt = np.load("/tmp/qtree_vectors_full.npy")
        with open("/tmp/gt_lookup_full.pkl", "rb") as f:
            gt = pickle.load(f)
        query_start = QUERY_START_FULL
        use_log1p = True
        cosine_ckpt = f"{BEST_MODEL_DIR}/best_compressor_full_fixed.pt"
        wj_ckpt = "/tmp/best_compressor_wj_native_full.pt"
        candidate_ks = candidate_ks_full
        n_epochs = epochs_full
    else:
        raise ValueError(name)

    corpus_qt = qt[:query_start]
    query_qt = qt[query_start:]
    corpus_sums = corpus_qt.sum(axis=1)
    print(f"qt={qt.shape} | corpus={corpus_qt.shape} | queries={query_qt.shape}")
    print(f"use_log1p={use_log1p} | ckpt={wj_ckpt}")
    return qt, gt, query_start, use_log1p, cosine_ckpt, wj_ckpt, candidate_ks, n_epochs, corpus_qt, query_qt, corpus_sums


qt, gt, query_start, use_log1p, cosine_ckpt, wj_ckpt, candidate_ks, n_epochs, corpus_qt, query_qt, corpus_sums = load_dataset(dataset_name)

qt=(10000, 18499) | corpus=(8000, 18499) | queries=(2000, 18499)
use_log1p=False | ckpt=/tmp/best_compressor_wj_native_10k.pt


In [13]:
# ── Train WJ-native MLP ──────────────────────────────────────────────────────
def train_wj_native(qt, gt, query_start, use_log1p, cosine_ckpt, wj_ckpt, n_epochs):
    model = QuadtreeCompressorWJ(qt.shape[1], out_dim=512, use_log1p=use_log1p).to(device)

    if init_from_cosine_ckpt and Path(cosine_ckpt).exists():
        state = torch.load(cosine_ckpt, map_location=device, weights_only=True)
        model.load_state_dict(state, strict=True)
        print(f"Warm-started from {cosine_ckpt}")
    else:
        print("Training from random init (no cosine checkpoint found).")

    if torch.cuda.device_count() > 1 and device.type == "cuda":
        model_par = nn.DataParallel(model)
        print(f"DataParallel on {torch.cuda.device_count()} GPUs")
    else:
        model_par = model

    dataset = AnchorPositiveDataset(qt, gt, query_start=query_start, max_pos=max_pos)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=device.type == "cuda",
        drop_last=True,
    )
    print(f"Steps/epoch: {len(loader)}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    best_loss = float("inf")
    history = []

    for epoch in range(1, n_epochs + 1):
        model_par.train()
        total_loss = 0.0
        total_steps = 0
        pbar = tqdm(loader, desc=f"Epoch {epoch:02d}/{n_epochs}", leave=False)

        for anchor, positive in pbar:
            anchor = anchor.to(device, non_blocking=True)
            positive = positive.to(device, non_blocking=True)
            B = anchor.shape[0]

            combined = torch.cat([anchor, positive], dim=0)
            out = model_par(combined)
            a_emb = out[:B]
            p_emb = out[B:]

            loss, n_viol, sim_ap, sim_an = wj_triplet_loss(a_emb, p_emb, margin=wj_margin)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += float(loss.detach().cpu())
            total_steps += 1
            pbar.set_postfix(loss=f"{float(loss):.4f}", viol=n_viol, sap=f"{float(sim_ap):.3f}")

        avg_loss = total_loss / max(total_steps, 1)
        scheduler.step()
        history.append(avg_loss)

        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), wj_ckpt)

        if epoch == 1 or epoch % 5 == 0 or epoch == n_epochs:
            print(
                f"Epoch {epoch:02d}/{n_epochs} | loss={avg_loss:.4f} | best={best_loss:.4f} | "
                f"lr={scheduler.get_last_lr()[0]:.2e}"
            )

    print(f"Saved best checkpoint: {wj_ckpt} (best_loss={best_loss:.4f})")
    return model, history


if run_training:
    model, train_history = train_wj_native(
        qt, gt, query_start, use_log1p, cosine_ckpt, wj_ckpt, n_epochs
    )
else:
    model = QuadtreeCompressorWJ(qt.shape[1], out_dim=512, use_log1p=use_log1p).to(device)
    model.load_state_dict(torch.load(wj_ckpt, map_location=device, weights_only=True))
    print(f"Loaded {wj_ckpt} (training skipped)")

model.eval()

Warm-started from /mnt/data1/ruban/hpmlproj/best_model/best_compressor_v1_clean.pt
Anchor-positive pairs: 46,722
Steps/epoch: 91


Epoch 01/50 | loss=0.3068 | best=0.3068 | lr=9.99e-04


Epoch 05/50 | loss=0.3017 | best=0.3017 | lr=9.76e-04


Epoch 10/50 | loss=0.3008 | best=0.3007 | lr=9.05e-04


Epoch 15/50 | loss=0.3007 | best=0.3005 | lr=7.94e-04


...

Epoch 35/50 | loss=0.3000 | best=0.2999 | lr=2.06e-04


Epoch 40/50 | loss=0.2999 | best=0.2999 | lr=9.55e-05


Epoch 45/50 | loss=0.2998 | best=0.2998 | lr=2.45e-05


Epoch 50/50 | loss=0.3000 | best=0.2998 | lr=0.00e+00
Saved best checkpoint: /tmp/best_compressor_wj_native_10k.pt (best_loss=0.2998)


QuadtreeCompressorWJ(
  (net): Sequential(
    (0): Linear(in_features=18499, out_features=4096, bias=False)
    (1): BatchNorm1d(4096, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=4096, out_features=1024, bias=False)
    (4): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): Linear(in_features=1024, out_features=512, bias=False)
    (7): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
)

In [14]:
# ── Eval utilities ───────────────────────────────────────────────────────────
def get_mem_mb():
    return psutil.Process(os.getpid()).memory_info().rss / 1024**2


def recall_at_k(gt_lookup, nbrs, query_start_id, k):
    total = 0.0
    count = 0
    for i, (ids, _) in enumerate(nbrs):
        qid = query_start_id + i
        gt_set = set(gt_lookup.get(qid, [])[:k])
        if not gt_set:
            continue
        total += len(gt_set & set(ids[:k])) / len(gt_set)
        count += 1
    return total / count if count else 0.0


def eval_recall(gt_lookup, nbrs, query_start_id, max_k):
    return {
        k: recall_at_k(gt_lookup, nbrs, query_start_id, k)
        for k in (10, 50, 100, 500)
        if k <= max_k
    }


def generate_wj_embeddings(model, data, device, batch_size=512):
    model.eval()
    chunks = []
    with torch.no_grad():
        for start in tqdm(range(0, len(data), batch_size), desc="Embedding (WJ)"):
            batch = torch.tensor(data[start:start + batch_size], dtype=torch.float32, device=device)
            emb = model(batch).cpu().numpy()
            chunks.append(emb)
    embs = np.vstack(chunks)
    assert np.all(embs >= -1e-6), "embeddings must be non-negative"
    sums = embs.sum(axis=1)
    print(f"Embedding simplex check: min_sum={sums.min():.4f} max_sum={sums.max():.4f}")
    return embs


def build_wj_index(corpus_embs):
    m0 = get_mem_mb()
    idx = nmslib.init(method="hnsw", space="WeightedJaccard")
    for i in tqdm(range(len(corpus_embs)), desc="Adding", mininterval=2.0):
        idx.addDataPoint(i, corpus_embs[i])
    t0 = time.time()
    idx.createIndex({"M": 20, "efConstruction": 200, "post": 1}, print_progress=True)
    build_s = time.time() - t0
    idx_mb = get_mem_mb() - m0
    idx.setQueryTimeParams({"efSearch": 200})
    return idx, build_s, idx_mb


def rerank_wj_gpu(query_qt, nbrs_raw, corpus_qt, corpus_sums, dev, batch_size=16):
    corpus_t = torch.from_numpy(corpus_qt).to(device=dev, dtype=torch.float32)
    corpus_sums_t = torch.from_numpy(corpus_sums).to(device=dev, dtype=torch.float32)
    reranked = [None] * len(nbrs_raw)
    for start in tqdm(range(0, len(nbrs_raw), batch_size), desc="GPU raw-WJ rerank"):
        batch = nbrs_raw[start:start + batch_size]
        groups = {}
        for offset, (ids, _) in enumerate(batch):
            ids_arr = np.asarray(ids, dtype=np.int64)
            groups.setdefault(len(ids_arr), []).append((start + offset, ids_arr))
        for cand_len, items in groups.items():
            if cand_len == 0:
                for absolute_i, _ in items:
                    reranked[absolute_i] = ([], [])
                continue
            ids_np = np.stack([ids for _, ids in items], axis=0)
            query_np = np.stack([query_qt[absolute_i] for absolute_i, _ in items], axis=0)
            ids_t = torch.from_numpy(ids_np).to(device=dev)
            q_t = torch.from_numpy(query_np).to(device=dev, dtype=torch.float32)
            c_t = corpus_t[ids_t]
            mins = torch.minimum(q_t[:, None, :], c_t).sum(dim=2)
            maxs = q_t.sum(dim=1, keepdim=True) + corpus_sums_t[ids_t] - mins
            order = torch.argsort(mins / maxs.clamp_min(1e-10), dim=1, descending=True).cpu().numpy()
            for row, (absolute_i, ids) in zip(order, items):
                reranked[absolute_i] = (ids[row].tolist(), [])
    del corpus_t, corpus_sums_t
    if dev.type == "cuda":
        torch.cuda.empty_cache()
    return reranked


def embedding_quality_check(model, qt, gt, query_start, device, n=200):
    """Compare raw-WJ vs embedding-WJ on GT pairs vs random pairs."""
    raw_gt, raw_rand, emb_gt, emb_rand = [], [], [], []
    qids = [q for q in gt if q >= query_start][:n]
    for qid in qids:
        pos_list = [p for p in gt[qid] if p < query_start]
        if not pos_list:
            continue
        pos_id = pos_list[0]
        rand_id = random.randrange(0, query_start)
        qv, pv, rv = qt[qid], qt[pos_id], qt[rand_id]
        raw_gt.append(weighted_jaccard_np(qv, pv))
        raw_rand.append(weighted_jaccard_np(qv, rv))
        with torch.no_grad():
            eq = model(torch.tensor(qv, dtype=torch.float32, device=device).unsqueeze(0))
            ep = model(torch.tensor(pv, dtype=torch.float32, device=device).unsqueeze(0))
            er = model(torch.tensor(rv, dtype=torch.float32, device=device).unsqueeze(0))
        emb_gt.append(float(wj_similarity(eq, ep).cpu()))
        emb_rand.append(float(wj_similarity(eq, er).cpu()))
    print(
        f"Raw WJ      — GT: {np.mean(raw_gt):.4f} | Rand: {np.mean(raw_rand):.4f} | "
        f"Gap: {np.mean(raw_gt) - np.mean(raw_rand):.4f}"
    )
    print(
        f"Emb WJ      — GT: {np.mean(emb_gt):.4f} | Rand: {np.mean(emb_rand):.4f} | "
        f"Gap: {np.mean(emb_gt) - np.mean(emb_rand):.4f}"
    )

In [15]:
# ── Retrieval evaluation ─────────────────────────────────────────────────────
def run_eval(model, qt, gt, query_start, corpus_qt, query_qt, corpus_sums, candidate_ks, label):
    print(f"\n{'=' * 72}")
    print(f"MLP WJ-NATIVE — {label}")
    print(f"{'=' * 72}")

    if Path(wj_ckpt).exists():
        model.load_state_dict(torch.load(wj_ckpt, map_location=device, weights_only=True))
        print(f"Eval checkpoint: {wj_ckpt}")
    else:
        print(f"Warning: {wj_ckpt} missing — using in-memory weights")
    model.eval()

    embedding_quality_check(model, qt, gt, query_start, device)

    embs = generate_wj_embeddings(model, qt, device)
    corpus_embs = embs[:query_start]
    query_embs = embs[query_start:]
    vec_mb = corpus_embs.nbytes / 1024**2
    print(f"embeddings={embs.shape} | corpus vec mem={vec_mb:.1f} MB")

    idx, build_s, idx_mb = build_wj_index(corpus_embs)
    print(f"WJ index build={build_s:.1f}s | idx mem={idx_mb:.1f} MB")

    results = {}
    max_k = max(max(candidate_ks), 500)

    print(f"\n--- Stage 1: WJ HNSW on {embs.shape[1]}-D embeddings (no rerank) ---")
    t0 = time.time()
    nbrs_nr = idx.knnQueryBatch(query_embs, k=max_k, num_threads=THREADS)
    qps_nr = len(query_embs) / (time.time() - t0)
    rec_nr = eval_recall(gt, nbrs_nr, query_start, max_k)
    results["wj_native_no_rerank"] = {
        **rec_nr,
        "qps": qps_nr,
        "build_s": build_s,
        "vec_mb": vec_mb,
        "idx_mb": idx_mb,
        "dim": embs.shape[1],
    }
    for k, r in rec_nr.items():
        print(f"  R@{k:<4} = {r:.4f}")
    print(f"  QPS={qps_nr:.1f}")

    for k in candidate_ks:
        print(f"\n--- Stage 2: top-{k} WJ candidates + exact raw-WJ rerank ({rerank_mode}) ---")
        t0 = time.time()
        nbrs_raw = idx.knnQueryBatch(query_embs, k=k, num_threads=THREADS)
        qps_cand = len(query_embs) / (time.time() - t0)

        if rerank_mode == "gpu":
            nbrs_rr = rerank_wj_gpu(
                query_qt, nbrs_raw, corpus_qt, corpus_sums, device, rerank_batch_size
            )
        else:
            raise ValueError("Only gpu rerank implemented in this notebook")

        rec_rr = eval_recall(gt, nbrs_rr, query_start, k)
        t1 = time.time()
        qps_total = len(query_embs) / (t1 - t0 + 1e-9)
        key = f"k{k}_raw_wj_rerank"
        results[key] = {**rec_rr, "qps": qps_total, "qps_candidates": qps_cand, "k": k}
        for rk, rv in rec_rr.items():
            print(f"  R@{rk:<4} = {rv:.4f}")
        print(f"  QPS (cand+rerank) ≈ {qps_total:.1f}")

    return results


all_results = {}
if run_eval:
    all_results[dataset_name] = run_eval(
        model,
        qt,
        gt,
        query_start,
        corpus_qt,
        query_qt,
        corpus_sums,
        candidate_ks,
        dataset_name,
    )

    payload = {}
    if Path(out_path).exists():
        with open(out_path, "rb") as f:
            payload = pickle.load(f)
    payload[dataset_name] = all_results[dataset_name]
    payload["_meta"] = {
        "train": "wj_triplet",
        "index": "WeightedJaccard",
        "wj_ckpt": wj_ckpt,
        "time": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    with open(out_path, "wb") as f:
        pickle.dump(payload, f)
    print(f"\nResults saved to {out_path}")

gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()


MLP WJ-NATIVE — 10k
Eval checkpoint: /tmp/best_compressor_wj_native_10k.pt
Raw WJ      — GT: 0.7996 | Rand: 0.1865 | Gap: 0.6131
Emb WJ      — GT: 0.9849 | Rand: 0.9423 | Gap: 0.0425


Embedding (WJ): 100%|██████████| 20/20 [00:00<00:00, 230.98it/s]


Embedding simplex check: min_sum=1.0000 max_sum=1.0000
embeddings=(10000, 512) | corpus vec mem=15.6 MB


Adding: 100%|██████████| 8000/8000 [00:00<00:00, 1316531.25it/s]

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
****************************************************



WJ index build=0.5s | idx mem=0.1 MB

--- Stage 1: WJ HNSW on 512-D embeddings (no rerank) ---
  R@10   = 0.3776
  R@50   = 0.5093
  R@100  = 0.5653
  R@500  = 0.7652
  QPS=18689.4

--- Stage 2: top-500 WJ candidates + exact raw-WJ rerank (gpu) ---


GPU raw-WJ rerank: 100%|██████████| 125/125 [00:00<00:00, 165.61it/s]


  R@10   = 0.9441
  R@50   = 0.8919
  R@100  = 0.8546
  R@500  = 0.7652
  QPS (cand+rerank) ≈ 2096.7

--- Stage 2: top-1000 WJ candidates + exact raw-WJ rerank (gpu) ---


GPU raw-WJ rerank: 100%|██████████| 125/125 [00:01<00:00, 116.86it/s]


  R@10   = 0.9660
  R@50   = 0.9310
  R@100  = 0.9049
  R@500  = 0.8350
  QPS (cand+rerank) ≈ 1578.9

Results saved to /tmp/results_mlp_wj_native.pkl


## Full dataset

1. Restart kernel (optional, frees GPU RAM).
2. Set `dataset_name = "full"` in the config cell.
3. Re-run all cells.

Compare with `02_mlp_cosine.ipynb` (cosine) and `03_neural_minhash.ipynb` (MinHash WJ).

In [16]:
# Quick comparison table (loads pickle if this notebook already ran)
compare = {
    "Baseline WJ (01)": "R@10 ~0.9966 (10k), ~0.9925 (full)",
    "MLP cosine no rerank (02)": "R@10 ~0.6655 (10k), ~0.6581 (full)",
    "Neural MinHash no rerank (03)": "R@10 ~0.6586 (10k), ~0.5821 (full)",
    "This notebook": out_path,
}
for name, ref in compare.items():
    print(f"{name}: {ref}")

if Path(out_path).exists():
    with open(out_path, "rb") as f:
        saved = pickle.load(f)
    for ds, runs in saved.items():
        if ds.startswith("_"):
            continue
        print(f"\n=== {ds} ===")
        for run_name, metrics in runs.items():
            r10 = metrics.get(10, metrics.get("10", float("nan")))
            qps = metrics.get("qps", float("nan"))
            print(f"  {run_name}: R@10={r10:.4f} QPS={qps:.1f}")

Baseline WJ (01): R@10 ~0.9966 (10k), ~0.9925 (full)
MLP cosine no rerank (02): R@10 ~0.6655 (10k), ~0.6581 (full)
Neural MinHash no rerank (03): R@10 ~0.6586 (10k), ~0.5821 (full)
This notebook: /tmp/results_mlp_wj_native.pkl

=== 10k ===
  wj_native_no_rerank: R@10=0.3776 QPS=18689.4
  k500_raw_wj_rerank: R@10=0.9441 QPS=2096.7
  k1000_raw_wj_rerank: R@10=0.9660 QPS=1578.9
